# Notebook 2 – Agent Integration

This notebook connects the knowledge base created in [Notebook 1](foundry-iq-knowledge-layer.ipynb) to a **Foundry Agent** using the **Model Context Protocol (MCP)**. By the end, you'll have:

1. A **project connection** that links your Foundry project to the knowledge base's MCP endpoint
2. An **agent** with an MCP tool that invokes `knowledge_base_retrieve` to answer questions
3. Multi-turn **conversations** that demonstrate knowledge-grounded responses with citations

## Architecture

```mermaid
flowchart TD
    User(("User Query")) --> Agent["Foundry Agent<br/><i>gpt-5.4-mini</i>"]
    Agent -->|"instructions: use KB tool"| MCP["MCPTool<br/><code>knowledge_base_retrieve</code>"]
    MCP --> Endpoint["Knowledge Base<br/>MCP Endpoint"]

    subgraph Retrieval["Agentic Retrieval Pipeline"]
        direction LR
        QP["Query Planning<br/><i>LLM decomposes query</i>"]
        S1["SearchIndex<br/>NASA Earth at Night"]
        S2["MCP Server<br/>Microsoft Learn"]
        QP --> S1 & S2
    end

    Endpoint --> Retrieval
    Retrieval --> Response["Agent synthesizes<br/>response with citations"]
    Response --> User

    style Agent fill:#f0fff4,stroke:#28a745,color:#000
    style Retrieval fill:#f0f9ff,stroke:#0078d4,color:#000
    style Response fill:#fffdf0,stroke:#d4a000,color:#000
```

## Prerequisites

- **Notebook 1 completed** — the knowledge base `foundry-iq-demo-kb` must exist on your search service
- **Microsoft Foundry project** with an LLM deployment (e.g., `gpt-5.4-mini`)
- **Foundry roles**: Foundry User (access models/agents), Foundry Project Manager (create project connections)
- Project's managed identity has **Search Index Data Reader** on the search service
- A `.env` file with `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_PROJECT_RESOURCE_ID`

## Phase 2A: Setup & Configuration

In [14]:
# Install required packages (uncomment if not already installed)
# %pip install azure-ai-projects>=2.0.0 requests azure-identity python-dotenv

In [15]:
import os
import requests
from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

load_dotenv()

# Azure AI Search (from Notebook 1)
search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]

# Microsoft Foundry Project
project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
project_resource_id = os.environ["FOUNDRY_PROJECT_RESOURCE_ID"]

# Agent model
agent_model = os.environ.get("LLM_DEPLOYMENT", "gpt-5.4-mini")

# Knowledge base name (must match Notebook 1)
KNOWLEDGE_BASE_NAME = "foundry-iq-demo-kb"

# Names for resources created in this notebook
PROJECT_CONNECTION_NAME = "foundry-iq-demo-mcp-connection"
AGENT_NAME = "foundry-iq-demo-agent"

# MCP endpoint for the knowledge base
MCP_ENDPOINT = (
    f"{search_endpoint}/knowledgebases/{KNOWLEDGE_BASE_NAME}"
    f"/mcp?api-version=2026-05-01-preview"
)

credential = DefaultAzureCredential()

print(f"Search endpoint:  {search_endpoint}")
print(f"Project endpoint: {project_endpoint}")
print(f"MCP endpoint:     {MCP_ENDPOINT}")
print(f"Agent model:      {agent_model}")


Search endpoint:  https://search-agentic-memory-ds.search.windows.net
Project endpoint: https://npmsfoundrysc.services.ai.azure.com/api/projects/proj-default
MCP endpoint:     https://search-agentic-memory-ds.search.windows.net/knowledgebases/foundry-iq-demo-kb/mcp?api-version=2026-05-01-preview
Agent model:      gpt-5.4-mini


## Phase 2B: Create Project Connection

A **project connection** of type `RemoteTool` tells Foundry Agent Service how to reach the knowledge base's MCP endpoint. The connection uses the project's **managed identity** to authenticate to Azure AI Search, so no API keys need to be stored.

We create this connection via the Azure Resource Manager REST API.

In [16]:
# Create a project connection to the knowledge base MCP endpoint
bearer_token_provider = get_bearer_token_provider(
    credential, "https://management.azure.com/.default"
)

headers = {
    "Authorization": f"Bearer {bearer_token_provider()}",
}

connection_url = (
    f"https://management.azure.com{project_resource_id}"
    f"/connections/{PROJECT_CONNECTION_NAME}?api-version=2025-10-01-preview"
)

connection_body = {
    "name": PROJECT_CONNECTION_NAME,
    "type": "Microsoft.MachineLearningServices/workspaces/connections",
    "properties": {
        "authType": "ProjectManagedIdentity",
        "category": "RemoteTool",
        "target": MCP_ENDPOINT,
        "isSharedToAll": True,
        "audience": "https://search.azure.com/",
        "metadata": {"ApiType": "Azure"},
    },
}

response = requests.put(connection_url, headers=headers, json=connection_body)
response.raise_for_status()

print(f"Connection '{PROJECT_CONNECTION_NAME}' created or updated successfully.")
print(f"  Target: {MCP_ENDPOINT}")
print(f"  Auth:   ProjectManagedIdentity")

Connection 'foundry-iq-demo-mcp-connection' created or updated successfully.
  Target: https://search-agentic-memory-ds.search.windows.net/knowledgebases/foundry-iq-demo-kb/mcp?api-version=2026-05-01-preview
  Auth:   ProjectManagedIdentity


## Phase 2C: Create Agent with MCP Tool

We create a Foundry Agent that integrates the knowledge base as an **MCP tool**. The agent:

- Uses instructions that explicitly direct it to call the knowledge base for all answers
- Is configured with `knowledge_base_retrieve` as its only allowed MCP tool
- Uses the project connection we just created for secure authentication

> **Note**: We use `allow_preview=True` on the `AIProjectClient` so we can get an agent-scoped OpenAI client via `get_openai_client(agent_name=...)`. This points the client directly at the agent's endpoint — no `extra_body` hacks needed.

In [17]:
# Create the project client (allow_preview enables agent-scoped OpenAI client)
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
    allow_preview=True,
)

# Agent instructions optimized for knowledge retrieval
AGENT_INSTRUCTIONS = """
You are a helpful assistant that must use the knowledge base to answer all questions.
You must never answer from your own knowledge under any circumstances.

Guidelines:
- Always call the knowledge base tool before responding.
- Provide clear, concise answers based on the retrieved content.
- Include citations to source documents using the annotation format.
- If the knowledge base does not contain the answer, respond with "I don't know based on the available sources."
- When information comes from multiple sources, synthesize them into a coherent response.
""".strip()

# Create the MCP tool pointing to the knowledge base
mcp_kb_tool = MCPTool(
    server_label="knowledge-base",
    server_url=MCP_ENDPOINT,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=PROJECT_CONNECTION_NAME,
)

# Create the agent
agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=agent_model,
        instructions=AGENT_INSTRUCTIONS,
        tools=[mcp_kb_tool],
    ),
)

print(f"Agent '{AGENT_NAME}' created successfully.")
print(f"  Model:   {agent_model}")
print(f"  Version: {agent.version}")
print(f"  Tools:   knowledge_base_retrieve (via MCP)")

Agent 'foundry-iq-demo-agent' created successfully.
  Model:   gpt-5.4-mini
  Version: 1
  Tools:   knowledge_base_retrieve (via MCP)


## Phase 2D: Invoke the Agent

We get an **agent-scoped OpenAI client** using `get_openai_client(agent_name=...)`, which points directly at the agent's endpoint. Multi-turn context is maintained by chaining `previous_response_id`.

When the agent receives a question, it:

1. Calls `knowledge_base_retrieve` via MCP
2. The knowledge base decomposes the query into subqueries
3. Subqueries run against both knowledge sources in parallel

4. Results are reranked and returned to the agent5. The agent synthesizes a natural-language response with citations

In [18]:
# Get an agent-scoped OpenAI client — points directly at the agent's endpoint
openai_client = project_client.get_openai_client(agent_name=AGENT_NAME)
print(f"Agent-scoped OpenAI client ready for: {AGENT_NAME}")


def display_response(response):
    """Display agent response with resolved citations."""
    print(f"\nAgent: {response.output_text}")

    # Extract and display citations from annotations
    citations = []
    for item in response.output:
        if item.type == "message":
            for content in item.content:
                if hasattr(content, "annotations") and content.annotations:
                    for ann in content.annotations:
                        if ann.type == "url_citation":
                            citations.append(
                                {"title": ann.title, "url": ann.url}
                            )
                        elif ann.type == "file_citation":
                            citations.append(
                                {"title": ann.filename, "file_id": ann.file_id}
                            )

    if citations:
        print(f"\n{'─' * 40}")
        print(f"SOURCES ({len(citations)}):")
        seen = set()
        for i, cite in enumerate(citations, 1):
            key = cite.get("url") or cite.get("file_id")
            if key in seen:
                continue
            seen.add(key)
            title = cite.get("title", "Untitled")
            if "url" in cite:
                print(f"  [{i}] {title}\n      {cite['url']}")
            else:
                print(f"  [{i}] {title}")
    else:
        print("\n  (No citations returned)")

Agent-scoped OpenAI client ready for: foundry-iq-demo-agent


In [19]:
# Query 1: Cross-source — should retrieve from both NASA data and Microsoft Learn
query1 = (
    "What patterns of light pollution can be observed from satellite imagery, "
    "and what Azure AI Search features support building retrieval systems "
    "over scientific document collections?"
)

print(f"User: {query1}")
print("-" * 80)

response1 = openai_client.responses.create(
    model=agent_model,
    input=query1,
)

display_response(response1)

User: What patterns of light pollution can be observed from satellite imagery, and what Azure AI Search features support building retrieval systems over scientific document collections?
--------------------------------------------------------------------------------

Agent: From the satellite imagery, the main light-pollution patterns are:

- Bright clusters over major urban and industrial regions, especially in North America, Europe, and East Asia【5:2†source】【5:4†source】
- Much darker areas over deserts, forests, oceans, and sparsely populated regions【5:2†source】【5:4†source】
- Linear lighting along transport or river corridors, such as the Indus River corridor and the Jane Addams Memorial Tollway near Chicago【5:8†source】【5:6†source】
- Offshore and remote light sources from fishing fleets, oil and gas activity, gas flares, and mining operations【5:5†source】【5:7†source】
- Changes over time, including more detailed urban lighting in later imagery and reduced brightness in conflict-affecte

In [20]:
# Query 2: Follow-up question — demonstrates conversational memory
query2 = "Can you tell me more about how holiday lights affect the seasonal patterns?"

print(f"User: {query2}")
print("-" * 80)

response2 = openai_client.responses.create(
    model=agent_model,
    input=query2,
    previous_response_id=response1.id,
)

display_response(response2)

User: Can you tell me more about how holiday lights affect the seasonal patterns?
--------------------------------------------------------------------------------

Agent: Yes. The retrieved material says holiday lights create a **seasonal increase in nighttime brightness**, especially in **December** in the United States. NASA researchers found U.S. nighttime lights can be **20 to 50 percent brighter in December**, and a related analysis of 70 cities found increases of **30 to 50 percent** in many places【9:7†source】【9:4†source】.

A few clear seasonal patterns show up:

- **Strongest increases in suburbs and outskirts of major cities**, where there are more single-family homes and yard space for decorations【9:8†source】
- **Smaller but still noticeable increases in central urban areas**, often around **20 to 30 percent** during the holidays【9:8†source】
- **Urban areas brighten more than rural areas**, with big metropolitan regions like Atlanta, Washington, D.C., Miami, Houston, and other

In [21]:
# Query 3: Microsoft Learn–focused question — should primarily hit the MCP source
query3 = "How do I create a knowledge source in Azure AI Search for agentic retrieval?"

print(f"User: {query3}")
print("-" * 80)

response3 = openai_client.responses.create(
    model=agent_model,
    input=query3,
    previous_response_id=response2.id,
)

display_response(response3)

User: How do I create a knowledge source in Azure AI Search for agentic retrieval?
--------------------------------------------------------------------------------

Agent: To create a knowledge source for agentic retrieval in Azure AI Search:

1. Go to your search service in the Azure portal.
2. Select **Agentic retrieval > Knowledge sources**.
3. Choose **Add knowledge source**.
4. Pick the source type, such as **Azure blob (Indexed)**.
5. Enter the source details, including the storage account and container.
6. If you want vector-based retrieval, enable **text vectorization** by adding a **vectorizer** and selecting your embedding model.
7. Save the vectorizer and then create the knowledge source【13:1†source】

A few prerequisites matter too:
- Your Azure AI Search service must be in a region that supports agentic retrieval【13:0†source】【13:2†source】
- For a search index–based knowledge source, the index must contain plain text or vector content plus a semantic configuration, and it mu

In [22]:
# Query 4: Out-of-scope question — should respond with "I don't know"
query4 = "What was the score of last night's basketball game?"

print(f"User: {query4}")
print("-" * 80)

response4 = openai_client.responses.create(
    model=agent_model,
    input=query4,
    previous_response_id=response3.id,
)

display_response(response4)

User: What was the score of last night's basketball game?
--------------------------------------------------------------------------------

Agent: I’m sorry, but I don’t have enough context to identify which basketball game you mean, and I can’t retrieve a specific score from the available information. If you tell me the teams, league, or date, I can help answer it.【17:0†source】

────────────────────────────────────────
SOURCES (1):
  [1] mcp://answersynthesis
      mcp://answersynthesis/


## Phase 2E: Cleanup

Uncomment and run the cells below to delete the agent and project connection.

> **Note**: This does not delete the knowledge base or knowledge sources (Notebook 1 resources). Delete those separately if needed.

In [23]:
# # Delete the agent
# project_client.agents.delete_version(
#     agent_name=agent.name,
#     agent_version=agent.version,
# )
# print(f"Agent '{agent.name}' version '{agent.version}' deleted.")

# # Delete the project connection
# bearer_token_provider = get_bearer_token_provider(
#     credential, "https://management.azure.com/.default"
# )
# headers = {"Authorization": f"Bearer {bearer_token_provider()}"}

# delete_url = (
#     f"https://management.azure.com{project_resource_id}"
#     f"/connections/{PROJECT_CONNECTION_NAME}?api-version=2025-10-01-preview"
# )
# response = requests.delete(delete_url, headers=headers)
# response.raise_for_status()
# print(f"Project connection '{PROJECT_CONNECTION_NAME}' deleted.")

## Summary

In this notebook, you:

1. **Created a project connection** linking the Foundry project to the knowledge base's MCP endpoint using managed identity authentication
2. **Created an agent** with an MCP tool (`knowledge_base_retrieve`) that queries the multi-source knowledge base
3. **Ran multi-turn conversations** demonstrating:
   - Cross-source retrieval (indexed data + live Microsoft Learn docs)
   - Conversational memory (follow-up questions)
   - Source-specific routing (Azure docs question → MCP source)
   - Graceful handling of out-of-scope questions

The agent transparently leverages Foundry IQ's agentic retrieval engine — query planning, parallel search across knowledge sources, semantic reranking, and citation-backed answers — all through a single MCP tool connection.